<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Fuwsegformer-inference/notebooks/3_uwsegformer_inference_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWSegFormer Offline Inference Test

Qualitative comparison of UWSegFormer predictions using checkpoints pretrained on SUIM and DUT over 5 internal project images.

## 1. Setup

In [ ]:
import os
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='matplotlib-'))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks' / '3_uwsegformer_inference_test.ipynb').exists():
            return candidate
    return start


# Override these with environment variables if your local/Colab layout differs.
REPO_ROOT = Path(os.environ.get('PROJECT_REPO_ROOT', find_repo_root())).resolve()
DRIVE_ROOT = Path(os.environ.get('UNDER_WATER_DATASET_ROOT', '/content/drive/MyDrive/Under_Water_Dataset'))

if IN_COLAB:
    PROJECT_TEST_DIR = Path(os.environ.get(
        'PROJECT_TEST_DIR',
        str(DRIVE_ROOT / 'Dataset_Processed' / 'Dataset_Final_512'),
    ))
    UWSEGFORMER_ROOT = Path(os.environ.get('UWSEGFORMER_ROOT', '/content/UWSegFormer'))
    CHECKPOINT_ROOT = Path(os.environ.get('CHECKPOINT_ROOT', str(DRIVE_ROOT / 'checkpoints')))
else:
    PROJECT_TEST_DIR = Path(os.environ.get(
        'PROJECT_TEST_DIR',
        str(REPO_ROOT / 'data' / 'SUIM' / 'images' / 'validation'),
    ))
    UWSEGFORMER_ROOT = Path(os.environ.get('UWSEGFORMER_ROOT', str(REPO_ROOT / 'external' / 'UWSegFormer')))
    CHECKPOINT_ROOT = Path(os.environ.get('CHECKPOINT_ROOT', str(REPO_ROOT / 'checkpoints')))

SUIM_CONFIG = UWSEGFORMER_ROOT / 'local_configs' / 'uwsegformer' / 'segformer_e+PPsegHead1.b0.640x480.suim.160k.py'
DUT_CONFIG = UWSEGFORMER_ROOT / 'local_configs' / 'uwsegformer' / 'segformer_e+PPsegHead1.b0.640x480.dut.40k.py'
SUIM_CHECKPOINT = CHECKPOINT_ROOT / 'uws_SUIM.pth'
DUT_CHECKPOINT = CHECKPOINT_ROOT / 'uws_dut.pth'

pd.Series({
    'IN_COLAB': IN_COLAB,
    'REPO_ROOT': str(REPO_ROOT),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'PROJECT_TEST_DIR': str(PROJECT_TEST_DIR),
    'UWSEGFORMER_ROOT': str(UWSEGFORMER_ROOT),
    'SUIM_CONFIG': str(SUIM_CONFIG),
    'DUT_CONFIG': str(DUT_CONFIG),
    'SUIM_CHECKPOINT': str(SUIM_CHECKPOINT),
    'DUT_CHECKPOINT': str(DUT_CHECKPOINT),
})


## 2. Select 5 Internal Test Images

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

candidate_images = sorted(
    path for path in PROJECT_TEST_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and 'mask' not in path.name.lower()
)

test_images = candidate_images[:5]
pd.DataFrame({'image_path': [str(path) for path in test_images]})

## 3. Load UWSegFormer Models

This cell assumes the selected UWSegFormer implementation is compatible with MMSegmentation's inference API. If the upstream repository exposes a custom inference function, replace this wrapper but keep the rest of the notebook unchanged.

In [ ]:
def load_mmseg_model(config_path: Path, checkpoint_path: Path, device: str = 'cuda:0'):
    from mmseg.apis import init_model

    if not config_path.exists():
        raise FileNotFoundError(f'Missing config: {config_path}')
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    return init_model(str(config_path), str(checkpoint_path), device=device)


def run_mmseg_inference(model, image_path: Path) -> np.ndarray:
    from mmseg.apis import inference_model

    result = inference_model(model, str(image_path))
    pred = result.pred_sem_seg.data.squeeze().detach().cpu().numpy()
    return pred.astype(np.uint8)


DEVICE = 'cuda:0'

suim_model = load_mmseg_model(SUIM_CONFIG, SUIM_CHECKPOINT, DEVICE)
dut_model = load_mmseg_model(DUT_CONFIG, DUT_CHECKPOINT, DEVICE)

## 4. Visual Comparison

In [ ]:
def read_rgb(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Could not read image: {image_path}')
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def colorize_mask(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(np.uint8)
    colored = cv2.applyColorMap((mask * 30) % 255, cv2.COLORMAP_TURBO)
    return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)


def plot_comparison(image_path: Path, suim_mask: np.ndarray, dut_mask: np.ndarray):
    image = read_rgb(image_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[1].imshow(colorize_mask(suim_mask))
    axes[1].set_title('UWSegFormer - SUIM weights')
    axes[2].imshow(colorize_mask(dut_mask))
    axes[2].set_title('UWSegFormer - DUT weights')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
for image_path in test_images:
    suim_prediction = run_mmseg_inference(suim_model, image_path)
    dut_prediction = run_mmseg_inference(dut_model, image_path)
    plot_comparison(image_path, suim_prediction, dut_prediction)

## 5. Qualitative Notes

| Image | SUIM checkpoint observation | DUT checkpoint observation | Preferred |
|---|---|---|---|
| 1 | TBD | TBD | TBD |
| 2 | TBD | TBD | TBD |
| 3 | TBD | TBD | TBD |
| 4 | TBD | TBD | TBD |
| 5 | TBD | TBD | TBD |